# Milestone 4: Training Runs on Kaggle

One notebook for every training run. List the runs to do in `RUN_IDS` (each is a file in `configs/`); they run one after another in this session, so a whole queue runs overnight. Then use **Save Version → Save & Run All**.

**Needs:** GPU (T4, or T4 ×2 for the final runs), Internet on, and the dataset `dsai-group3-m4-data` attached. Keep a queue under about 11 hours: a Kaggle session stops at 12.

In [ ]:
RUN_IDS = ["default"]     # e.g. ["T-02", "T-03", "T-04"]; each is configs/<RUN_ID>.yaml
BRANCH = "milestone-4"
RESUME = False            # True to continue the first run in the list from its last checkpoint

In [ ]:
!git clone -q --depth 1 -b {BRANCH} https://github.com/thehimanshhu/Group-3-DS-and-AI-Lab-Project repo
!pip install -q -r repo/requirements.txt
!git -C repo rev-parse --short HEAD

In [ ]:
from pathlib import Path

manifests = list(Path('/kaggle/input').rglob('manifest_m4.csv'))
assert manifests, 'Attach the dataset dsai-group3-m4-data'
DATA_DIR = manifests[0].parent.parent
missing = [r for r in RUN_IDS if not Path(f'repo/configs/{r}.yaml').exists()]
assert not missing, f'No config for {missing}: merge it into {BRANCH} first'
print(DATA_DIR, '| queue:', RUN_IDS)

In [ ]:
import subprocess, time

for i, run_id in enumerate(RUN_IDS):
    out_dir = Path('/kaggle/working') / run_id
    cmd = ['python', '-m', 'src.train', '--config', f'configs/{run_id}.yaml', '--data', str(DATA_DIR), '--out', str(out_dir)]
    if RESUME and i == 0:
        cmd.append('--resume')
    start = time.time()
    result = subprocess.run(cmd, cwd='repo')
    print(f'{run_id}: exit code {result.returncode}, {(time.time() - start) / 60:.0f} min')
    # A failed run doesn't stop the queue; its folder shows what happened.

In [ ]:
import json
import pandas as pd

rows = [json.load(open(p)) for run_id in RUN_IDS if (p := Path('/kaggle/working') / run_id / 'run.json').exists()]
pd.DataFrame(rows)

**After the queue:**

1. For each run, download the small files from this notebook's output (`run.json`, `config.yaml`, `train_log.csv`, `epoch_log.csv`, `predictions.csv`, `results.json`) into `results/<RUN_ID>/` on your own branch and open a PR.
2. `best.pt` stays here, in the notebook output. Share this notebook with the team so it can be attached as an input.
3. Tick the runs on the "M4 run tracker" issue and post their `valid` EER on WhatsApp.